In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import json
import random
from typing import Dict, List, Optional

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef


def get_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def load_clean_dataset(base_dir: str, dataset_name: str) -> pd.DataFrame:
    csv_path = os.path.join(base_dir, "clean", f"{dataset_name}_clean.csv")
    return pd.read_csv(csv_path).reset_index(drop=True)


def encode_texts_for_dataset(df, tokenizer, transformer, device, max_length=128, batch_size=32):
    texts = df["text"].astype(str).tolist()
    all_embeddings = []
    transformer.eval()
    with torch.no_grad():
        batches = int(np.ceil(len(texts) / batch_size))
        for i in tqdm(range(batches), desc="  Encoding texts"):
            batch_texts = texts[i * batch_size:(i + 1) * batch_size]
            enc = tokenizer(batch_texts, padding=True, truncation=True, max_length=max_length, return_tensors="pt")
            enc = {k: v.to(device) for k, v in enc.items()}
            outputs = transformer(**enc)
            cls_emb = outputs.last_hidden_state[:, 0, :]
            all_embeddings.append(cls_emb.cpu())
    return torch.cat(all_embeddings, dim=0)


def prepare_all_datasets(base_dir, model_name, max_length, batch_size, device, dataset_names):
    print(f"Using device: {device}")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    transformer = AutoModel.from_pretrained(model_name).to(device)
    all_data = {}
    for name in dataset_names:
        print(f"\n=== Preparing dataset: {name} ===")
        df = load_clean_dataset(base_dir, name)
        print(f"  Num documents: {len(df)}")
        embeddings = encode_texts_for_dataset(df, tokenizer, transformer, device, max_length, batch_size)
        labels = torch.tensor(df["label"].values, dtype=torch.long)
        split = df["split"].values
        train_mask = torch.tensor(split == "train", dtype=torch.bool)
        val_mask = torch.tensor(split == "val", dtype=torch.bool)
        test_mask = torch.tensor(split == "test", dtype=torch.bool)
        all_data[name] = {"emb": embeddings, "labels": labels, "train_mask": train_mask,
                           "val_mask": val_mask, "test_mask": test_mask}
    return all_data


class TextOnlyClassifier(nn.Module):
    def __init__(self, emb_dim, hidden=256, num_classes=2, dropout=0.3):
        super().__init__()
        self.fc1 = nn.Linear(emb_dim, hidden)
        self.fc2 = nn.Linear(hidden, num_classes)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        h = F.relu(self.fc1(x))
        h = self.dropout(h)
        return self.fc2(h)


def resample_train_mask(labels, orig_train_mask, target_fake_ratio, seed):
    rng = np.random.RandomState(seed)
    train_idx = torch.nonzero(orig_train_mask, as_tuple=True)[0].numpy()
    train_labels = labels[orig_train_mask].numpy()
    fake_idx = train_idx[train_labels == 1]
    real_idx = train_idx[train_labels == 0]
    Fk, R = len(fake_idx), len(real_idx)
    p = target_fake_ratio
    new_fake_a = Fk
    new_real_a = round(Fk * (1 - p) / p) if p > 0 else R
    new_real_b = R
    new_fake_b = round(R * p / (1 - p)) if p < 1 else Fk
    if new_real_a <= R:
        new_fake, new_real = new_fake_a, new_real_a
    else:
        new_fake, new_real = new_fake_b, new_real_b
    new_fake = max(1, min(new_fake, Fk))
    new_real = max(1, min(new_real, R))
    chosen_fake = rng.choice(fake_idx, size=new_fake, replace=False)
    chosen_real = rng.choice(real_idx, size=new_real, replace=False)
    chosen = np.concatenate([chosen_fake, chosen_real])
    new_mask = torch.zeros_like(orig_train_mask)
    new_mask[chosen] = True
    counts = torch.tensor([new_real, new_fake], dtype=torch.float32)  # [count_class0, count_class1]
    return new_mask, counts


# ================================================================
# Loss functions. All per-class weight vectors are normalized so
# mean(weight) == 1 (i.e. weights.sum() == num_classes), so that
# overall loss magnitude is comparable across weighting SCHEMES --
# isolating differences in weighting *shape*, not scale.
# ================================================================
def normalize_weights(weights):
    return weights / weights.mean()


def inverse_frequency_weights(counts, device):
    n = counts.sum()
    num_classes = len(counts)
    w = n / (num_classes * counts.clamp(min=1.0))
    return normalize_weights(w).to(device)


def class_balanced_weights(counts, beta, device):
    counts = counts.clamp(min=1.0)
    effective_num = (1 - torch.pow(beta, counts)) / (1 - beta)
    w = 1.0 / effective_num
    return normalize_weights(w).to(device)


def focal_loss(logits, targets, alpha_weight, gamma=2.0):
    logpt = F.log_softmax(logits, dim=1)
    pt = logpt.exp()
    logpt_t = logpt.gather(1, targets.unsqueeze(1)).squeeze(1)
    pt_t = pt.gather(1, targets.unsqueeze(1)).squeeze(1)
    if alpha_weight is not None:
        alpha_t = alpha_weight[targets]
    else:
        alpha_t = 1.0
    loss = -alpha_t * (1 - pt_t).pow(gamma) * logpt_t
    return loss.mean()


def balanced_softmax_loss(logits, targets, class_counts):
    logits_adj = logits + torch.log(class_counts.clamp(min=1.0))
    return F.cross_entropy(logits_adj, targets)


def compute_loss(method, logits, targets, train_counts, device):
    if method == "unweighted_ce":
        return F.cross_entropy(logits, targets)
    elif method == "weighted_ce":
        w = inverse_frequency_weights(train_counts, device)
        return F.cross_entropy(logits, targets, weight=w)
    elif method == "weighted_focal":
        w = inverse_frequency_weights(train_counts, device)
        return focal_loss(logits, targets, w, gamma=2.0)
    elif method == "unweighted_focal":
        return focal_loss(logits, targets, None, gamma=2.0)
    elif method == "balanced_softmax":
        return balanced_softmax_loss(logits, targets, train_counts.to(device))
    elif method.startswith("class_balanced_"):
        beta = float(method.split("_")[-1])
        w = class_balanced_weights(train_counts, beta, device)
        return F.cross_entropy(logits, targets, weight=w)
    else:
        raise ValueError(method)


def lr_lambda(epoch, total_epochs, warmup_frac=0.1):
    warmup_epochs = max(1, int(total_epochs * warmup_frac))
    if epoch < warmup_epochs:
        return (epoch + 1) / warmup_epochs
    progress = (epoch - warmup_epochs) / max(1, total_epochs - warmup_epochs)
    return 0.5 * (1 + np.cos(np.pi * progress))


def _pred_fake_frac_and_collapse(pred, threshold):
    frac = pred.float().mean().item()
    collapsed = (frac >= threshold) or (frac <= (1 - threshold))
    max_skew = max(frac, 1 - frac)  # how far toward EITHER extreme, regardless of direction
    return frac, collapsed, max_skew


def train_one_run(
    method, src_name, all_data, device, target_fake_ratio, seed,
    epochs=30, lr=1e-3, weight_decay=1e-4, hidden=256, dropout=0.3,
    grad_clip_norm=1.0, collapse_threshold=0.98,
):
    """
    Full-batch throughout (matches the rest of this paper's primary experimental
    regime). `seed` controls BOTH data resampling and model initialization
    identically across all methods -- i.e. for a fixed (src, ratio, seed), every
    method trains on the exact same resampled examples starting from the exact
    same initial weights, differing ONLY in loss function.

    Checkpoint accounting (fixes applied per reviewer/co-author feedback):
      - The TRUE untrained (zero gradient steps) checkpoint is evaluated and is a
        real candidate for "best" -- distinct from "epoch 0", which (in every
        other experiment in this paper) actually means "after one gradient step".
        We record it as epoch index -1 if it wins checkpoint selection.
      - `collapsed_final` is computed by RE-EVALUATING the RESTORED best
        checkpoint on validation data, not by reading off the last training
        epoch's cached validation curve value (these can disagree whenever the
        best checkpoint isn't the final epoch).
      - `ever_collapsed` (crossed the threshold at any point, including the
        pretrain checkpoint) is tracked separately from `collapsed_final`
        (true state of the checkpoint actually used for evaluation), plus
        `max_pred_fake_frac_skew` (the most extreme skew reached at any point)
        so transient and persistent collapse are distinguishable in the saved
        data, not conflated into one flag.
    """
    data_src = all_data[src_name]
    x_src = data_src["emb"].to(device)
    y_src = data_src["labels"].to(device)
    val_mask = data_src["val_mask"].to(device)
    test_mask = data_src["test_mask"].to(device)

    train_mask, train_counts = resample_train_mask(data_src["labels"], data_src["train_mask"], target_fake_ratio, seed)
    train_mask = train_mask.to(device)

    set_seed(seed)
    emb_dim = x_src.size(1)
    model = TextOnlyClassifier(emb_dim, hidden, 2, dropout).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lambda ep: lr_lambda(ep, epochs))

    # ---- Evaluate the TRUE untrained checkpoint (zero gradient steps) first ----
    model.eval()
    with torch.no_grad():
        pretrain_val_logits = model(x_src)[val_mask]
        pretrain_val_pred = pretrain_val_logits.argmax(dim=-1)
        pretrain_val_f1 = f1_score(y_src[val_mask].cpu().numpy(), pretrain_val_pred.cpu().numpy(), average="macro")
    pretrain_frac, pretrain_collapsed, pretrain_skew = _pred_fake_frac_and_collapse(pretrain_val_pred, collapse_threshold)

    best_val_f1 = pretrain_val_f1
    best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    best_checkpoint_epoch = -1  # -1 == truly untrained, distinct from "epoch 0" (after 1 step)

    # Three-way split (per co-author feedback) instead of one conflated flag:
    #   pretrain_collapsed      -- was the model ALREADY skewed at random init, before
    #                              any gradient step? (an initialization artifact, not
    #                              an optimization-induced failure)
    #   training_ever_collapsed -- did the threshold get crossed at any point DURING
    #                              actual training (epoch >= 1), regardless of init state?
    #   collapsed_final         -- is the RESTORED, evaluated checkpoint collapsed?
    pretrain_collapsed_flag = pretrain_collapsed
    training_ever_collapsed = False
    max_skew_seen = pretrain_skew
    epoch_first_collapse = None  # among TRAINED epochs only (0-indexed, matches prior experiments)

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        logits = model(x_src)
        loss = compute_loss(method, logits[train_mask], y_src[train_mask], train_counts, device)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip_norm)
        optimizer.step()
        scheduler.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(x_src)[val_mask]
            val_pred = val_logits.argmax(dim=-1)
            val_f1 = f1_score(y_src[val_mask].cpu().numpy(), val_pred.cpu().numpy(), average="macro")
        pred_fake_frac, is_collapsed_now, skew_now = _pred_fake_frac_and_collapse(val_pred, collapse_threshold)

        training_ever_collapsed = training_ever_collapsed or is_collapsed_now
        max_skew_seen = max(max_skew_seen, skew_now)
        if is_collapsed_now and epoch >= 1 and epoch_first_collapse is None:
            epoch_first_collapse = epoch

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            best_checkpoint_epoch = epoch

    # ---- Restore the SELECTED best checkpoint and re-evaluate everything from it ----
    model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
    model.eval()
    with torch.no_grad():
        restored_val_logits = model(x_src)[val_mask]
        restored_val_pred = restored_val_logits.argmax(dim=-1)
        restored_val_labels = y_src[val_mask]
        restored_val_f1 = f1_score(restored_val_labels.cpu().numpy(), restored_val_pred.cpu().numpy(), average="macro")
        restored_val_mcc = matthews_corrcoef(restored_val_labels.cpu().numpy(), restored_val_pred.cpu().numpy())

        test_logits = model(x_src)[test_mask]
        test_pred = test_logits.argmax(dim=-1)
        test_labels = y_src[test_mask]
        test_acc = accuracy_score(test_labels.cpu().numpy(), test_pred.cpu().numpy())
        test_f1 = f1_score(test_labels.cpu().numpy(), test_pred.cpu().numpy(), average="macro")
        test_mcc = matthews_corrcoef(test_labels.cpu().numpy(), test_pred.cpu().numpy())

    # collapsed_final is computed from the RESTORED checkpoint's own validation
    # predictions -- not the last training epoch's cached curve value.
    restored_val_frac, collapsed_final, restored_skew = _pred_fake_frac_and_collapse(restored_val_pred, collapse_threshold)
    test_pred_fake_frac = test_pred.float().mean().item()

    return {
        "method": method, "source": src_name, "target_fake_ratio": target_fake_ratio, "seed": seed,
        "best_checkpoint_epoch": best_checkpoint_epoch,  # -1 = truly untrained beat every trained epoch
        "epoch_first_collapse": epoch_first_collapse if epoch_first_collapse is not None else -1,
        "pretrain_collapsed": bool(pretrain_collapsed_flag),        # skewed at random init, pre-gradient-step
        "training_ever_collapsed": bool(training_ever_collapsed),   # crossed threshold during actual training
        "collapsed_final": bool(collapsed_final),  # from RESTORED best checkpoint, matches what's evaluated
        "max_pred_fake_frac_skew": max_skew_seen,  # most extreme skew reached at any point in training
        "restored_val_pred_fake_frac": restored_val_frac,
        "restored_val_macro_f1": restored_val_f1,
        "restored_val_mcc": restored_val_mcc,
        "test_macro_f1": test_f1, "test_acc": test_acc, "test_mcc": test_mcc,
        "test_pred_fake_frac": test_pred_fake_frac,
        "best_val_macro_f1": best_val_f1,
    }


def run_mitigation_comparison(base_dir, sources=["coaid", "fakenewsnet"], model_name="roberta-base",
                               epochs=30, max_length=128, dropout=0.3):
    device = get_device()
    all_data = prepare_all_datasets(base_dir, model_name, max_length, 32, device, sources)

    out_dir = os.path.join(base_dir, "mitigation_comparison_results")
    os.makedirs(out_dir, exist_ok=True)

    main_ratios = [0.50, 0.60, 0.70, 0.80, 0.85]
    main_methods = ["unweighted_ce", "weighted_ce", "weighted_focal", "balanced_softmax", "class_balanced_0.9999"]
    main_seeds = list(range(5))

    supplement_focal_ratios = [0.70, 0.80, 0.85]
    supplement_cb_ratios = [0.70, 0.85]
    supplement_cb_betas = [0.99, 0.999]

    jobs = []
    for src in sources:
        for ratio in main_ratios:
            for seed in main_seeds:
                for method in main_methods:
                    jobs.append((src, ratio, seed, method))
    for src in sources:
        for ratio in supplement_focal_ratios:
            for seed in main_seeds:
                jobs.append((src, ratio, seed, "unweighted_focal"))
    for src in sources:
        for ratio in supplement_cb_ratios:
            for seed in main_seeds:
                for beta in supplement_cb_betas:
                    jobs.append((src, ratio, seed, f"class_balanced_{beta}"))

    print(f"\n=== Mitigation comparison: {len(jobs)} total runs "
          f"({len(sources)*len(main_ratios)*len(main_seeds)*len(main_methods)} main + "
          f"{len(jobs) - len(sources)*len(main_ratios)*len(main_seeds)*len(main_methods)} supplemental) ===")

    rows = []
    for counter, (src, ratio, seed, method) in enumerate(jobs, 1):
        result = train_one_run(method, src, all_data, device, ratio, seed, epochs=epochs, dropout=dropout)
        if counter % 25 == 0 or counter == len(jobs):
            print(f"[{counter}/{len(jobs)}] {src} ratio={ratio} seed={seed} method={method} "
                  f"-> F1={result['test_macro_f1']:.4f} collapsed={result['collapsed_final']}")
        rows.append(result)

    df = pd.DataFrame(rows)
    out_path = os.path.join(out_dir, "mitigation_comparison_summary.csv")
    df.to_csv(out_path, index=False)
    print(f"\nSaved to: {out_path}")

    split_cols = ["pretrain_collapsed", "training_ever_collapsed", "collapsed_final"]

    print("\n=== Collapse rate by method (main grid, all ratios pooled): "
          "pretrain_collapsed / training_ever_collapsed / collapsed_final ===")
    main_df = df[df["method"].isin(main_methods)]
    print(main_df.groupby(["method"])[split_cols].agg(["mean", "count"]))

    print("\n=== collapsed_final rate by method x ratio (main grid) -- THE headline result ===")
    pivot = main_df.groupby(["method", "target_fake_ratio"])["collapsed_final"].mean().unstack()
    print(pivot.reindex(index=main_methods))

    print("\n=== pretrain_collapsed rate by method x ratio -- initialization artifact, "
          "should be roughly constant across methods since it happens before any "
          "method-specific loss is even applied ===")
    pivot_pretrain = main_df.groupby(["method", "target_fake_ratio"])["pretrain_collapsed"].mean().unstack()
    print(pivot_pretrain.reindex(index=main_methods))

    print("\n=== training_ever_collapsed rate by method x ratio -- TRANSIENT collapse "
          "during actual optimization, separate from initialization ===")
    pivot_train_ever = main_df.groupby(["method", "target_fake_ratio"])["training_ever_collapsed"].mean().unstack()
    print(pivot_train_ever.reindex(index=main_methods))

    print("\n=== Mean max_pred_fake_frac_skew by method (main grid) -- "
          "how close to total collapse each method got, even if it recovered ===")
    print(main_df.groupby("method")["max_pred_fake_frac_skew"].agg(["mean", "max"]).reindex(main_methods))

    print("\n=== best_checkpoint_epoch distribution by method (main grid) -- "
          "-1 = truly untrained checkpoint won selection ===")
    print(main_df.groupby("method")["best_checkpoint_epoch"].apply(
        lambda s: (s == -1).mean()
    ).rename("frac_selected_untrained_checkpoint"))

    print("\n=== Supplement: unweighted_focal -- 3-way split by ratio ===")
    supp_focal = df[df["method"] == "unweighted_focal"]
    print(supp_focal.groupby("target_fake_ratio")[split_cols].agg(["mean", "count"]))

    print("\n=== Supplement: class_balanced beta-sensitivity (0.99 / 0.999 / 0.9999) -- 3-way split ===")
    cb_rows = df[df["method"].str.startswith("class_balanced_")]
    print(cb_rows.groupby(["method", "target_fake_ratio"])[split_cols].agg(["mean", "count"]))

    print("\n=== MCC summary by method (main grid, test set, mean +/- std) ===")
    print(main_df.groupby("method")["test_mcc"].agg(["mean", "std"]).reindex(main_methods))

    return df


run_mitigation_comparison(
    base_dir="drive/MyDrive/CSML/data",
    sources=["coaid", "fakenewsnet"],
    model_name="roberta-base",
    epochs=30,
    max_length=128,
    dropout=0.3,
)
